# 09 · Filter genes, cells and guides

Reduces the single-knockout object to the genes, cells and guides the effect
models are fitted on.

**Reads** `par_save_filename_5` and `par_outlier_controlguides_file`.
**Writes** `par_save_filename_7`.

The gene filter runs before the cell filter, so the detected-gene count each
cell is judged on is measured over the reduced gene set. Reversing the order
changes which cells survive.

## Setup

In [1]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)

In [2]:
adata = sc.read(par_save_filename_5)
print(f"input: {adata.shape[0]} cells x {adata.shape[1]} genes")

sc.pp.filter_genes(adata, min_cells=par_mincells_for_testedgenes)
sc.pp.filter_cells(adata, min_genes=par_mincellgenes_for_testedgenes)
print(f"after gene/cell filtering: {adata.shape[0]} cells x {adata.shape[1]} genes")

input: 341664 cells x 13811 genes
filtered out 7126 genes that are detected in less than 20000 cells
filtered out 13007 cells that have less than 800 genes expressed
after gene/cell filtering: 328657 cells x 6685 genes


## Drop rare guides and non-inert control guides

A guide seen in too few cells cannot support an effect estimate. The outlier
control guides come from notebook 08.

In [3]:
guide_names = list(adata.uns["feature_barcode_names"])
carried = adata.obs[guide_names] > 0

outlier_controls = set(pd.read_csv(par_outlier_controlguides_file)["OutlierGuides"])
rare = carried.sum(axis=0) < par_ncell_test_threshold
drop = rare | carried.columns.isin(outlier_controls)

print(f"guides dropped as rare              : {int(rare.sum())}")
print(f"guides dropped as outlier controls  : {len(outlier_controls)}")

kept_guides = [g for g in guide_names if not drop[g]]
adata.uns["feature_barcode_names_filtered"] = kept_guides
adata.uns["feature_KO_barcode_names_filtered"] = [
    g for g in kept_guides
    if not g.startswith((par_not_target_control_prefix, par_nongene_site_control_prefix))
]
print(f"guides kept: {len(kept_guides)} "
      f"({len(adata.uns['feature_KO_barcode_names_filtered'])} knockout)")

guides dropped as rare              : 190
guides dropped as outlier controls  : 31
guides kept: 3502 (3204 knockout)


## Keep cells that still carry a guide, and write

In [4]:
adata = adata[(adata.obs[kept_guides] > 0).sum(axis=1) > 0].copy()
print(f"cells still carrying a guide: {adata.shape[0]}")

adata.write(par_save_filename_7)
print(f"written: {par_save_filename_7}  ({adata.shape[0]} x {adata.shape[1]})")

cells still carrying a guide: 325203
written: outputs/anndata/adata-SingleKO_Filtered.h5ad  (325203 x 6685)
